# EVA-02 Base - model reference

Weights download automatically from Hugging Face Hub on first run and are cached locally afterward.

This notebook documents the model architecture: block structure, token counts, pooling behaviour, etc. used in other parts of the project. It feeds nothing downstream - `03_export_pipeline.ipynb` derives everything it needs (data config, feature dimensionality) directly from the loaded model rather than importing anything from here. This notebook was created to inspect the model architecture for the subject Principles of Deep Learning (Principi dubokog ucenja), which was the basis for further project development of Statsictical Prigramming. 

In [ ]:
import timm
import torch

In [ ]:
import logging
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)  # silences the "unauthenticated requests" notice -- harmless, just a rate-limit tip

# Load the pretrained EVA-02 Base model with its 1000-class classification head
model_name = "eva02_base_patch14_448.mim_in22k_ft_in22k_in1k"
model = timm.create_model(model_name, pretrained=True)
model.eval()

**Model architecture:**<br>
- 12-block ViT-style network,<br>
- SwiGLU-gated MLPs (not plain GELU),<br>
- rotary position embeddings rather than learned absolute ones,<br>
- and a 14x14-pixel patch embedding into a 768-d token space.<br>

This printout is the reference for the block count and component names referred to elsewhere in the project.

In [ ]:
# Build the preprocessing pipeline that matches what this model expects
data_config = timm.data.resolve_model_data_config(model)
transform = timm.data.create_transform(**data_config, is_training=False)

**What does the checkpoint expect as input, and where does it run?** `resolve_model_data_config` reads the preprocessing this specific checkpoint was trained with (448x448, 3-channel).<br>
`Using device: cpu` is expected on this machine: the installed torch build has no CUDA support even though a GPU is present, so extraction runs on CPU throughout this project.

In [ ]:
# Confirm the model loaded correctly
input_size = data_config["input_size"]
print(f"Model loaded: {model_name}")
print(f"Expected input size : {input_size[1]}x{input_size[2]} px  (channels: {input_size[0]})")
print(f"Number of output classes: {model.num_classes}")
print(f"Using device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

In [ ]:
print(model.global_pool)

**How does the model turn many patch tokens into one feature vector?** `avg` means simple mean-pooling across tokens, not just taking the CLS token as the summary.

In [ ]:
print(model.num_prefix_tokens)   # 1 means CLS is treated as a prefix (typically excluded from avg pool)

**By returning 1** `num_prefix_tokens = 1` confirms the single CLS token is treated as a prefix and excluded from the average-pool checked earlier. Put together: the 768-d pre-logits vector is the mean of the 1,024 patch-token embeddings, CLS excluded.

In [ ]:
dummy = torch.randn(1, 3, 448, 448)   # 1 image, 3 channels, 448×448 — matches your config

model.eval()
with torch.no_grad():
    feats  = model.forward_features(dummy)                 # expect [1, N, 768]
    pooled = model.forward_head(feats, pre_logits=True)    # expect [1, 768]

print(feats.shape, pooled.shape)

**How many tokens does a 448x448 image actually produce?** 1,025 = 1,024 patch tokens (448/14 = 32 patches per side, 32² = 1,024) plus one CLS token;<br>
`forward_head(..., pre_logits=True)` then collapses that down to the single 768-d vector used everywhere downstream.